# Session 3 — the benchmark

Ten small pure-Python repos, each audited three ways: **near** injection, **far**
injection, and **clean** (negative control). ~30 audits, ~45 min.

Cell 2 is a **CPU-only preflight** — it downloads, injects, and verifies every
planted line resolves through the manifest. Run it before booking GPU time; a dead
repo or broken injection costs seconds there instead of 45 minutes later.

**T4 GPU → Runtime → Restart session before running.**

In [ ]:
#@title 1. Install — version check
import sys
assert "wca" not in sys.modules, "Stale module. Runtime > Restart session."

REPO = "https://github.com/quinyang/whole_codebase_auditor"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}

!pip install -q --upgrade --force-reinstall --no-deps "wca @ git+{REPO}@{BRANCH}"
!pip install -q "wca[gpu] @ git+{REPO}@{BRANCH}"

import wca
print("wca", wca.__version__)
MIN = (0, 9, 0)
assert tuple(map(int, wca.__version__.split("."))) >= MIN, (
    f"got {wca.__version__}, need >= 0.9.0 -- the push did not land"
)

In [ ]:
#@title 2. CPU preflight — build and validate the corpus (~10 s, no GPU)
from wca.benchmark import prepare_synthetic_corpus

corpus = prepare_synthetic_corpus(budget=4000)

# Generated, not downloaded. Measured on a T4: the budget gives the packer
# ~1,400 tokens of full bodies, while a single 350-line module from a real
# library is ~5,600 tokens -- both halves of a cross-file defect physically
# cannot be in context. See wca/corpus.py. The 'coverage' column is what the
# packer would surface at whole-repo scale; report it alongside detection.

In [ ]:
#@title 3. Run the benchmark (~45 min; checkpoints after every audit)
from wca.infer import load_auditor
from wca.benchmark import run_benchmark

auditor = load_auditor()   # cached: re-running this cell reuses the model      # ~6 GiB allocated means 4-bit worked
outcomes, summary = run_benchmark(corpus, auditor=auditor, budget=4000)

In [ ]:
#@title 4. Plot near vs far recall
import matplotlib.pyplot as plt
bv = summary["by_variant"]
names = [v for v in ("near", "far") if bv[v]["recall"] is not None]
if names:
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.bar(names, [bv[v]["recall"] for v in names], color=["#3b7dd8", "#d87f3b"])
    ax.set(ylabel="recall", ylim=(0, 1),
           title="Cross-file recall vs separation\nin the packed stream")
    for i, v in enumerate(names):
        ax.text(i, bv[v]["recall"] + .02, f"{bv[v]['recall']:.0%}", ha="center")
    ax.grid(axis="y", alpha=.3)
    plt.tight_layout(); plt.savefig("/content/wca_runs/ablation.png", dpi=140); plt.show()
print(f"precision {summary['precision']:.1%} | recall {summary['recall']:.1%} "
      f"| F1 {summary['f1']:.3f}")

## Reporting this honestly

- Quote precision/recall over **grounded findings only**, and say so.
- Report the negative-control line: how many findings were proposed on clean
  repos and how many survived grounding. That ratio is the mechanism.
- Report the near/far delta **whatever it is**. A null result on your own
  ablation is more credible than a claim, and reviewers ask about ablations.
- State the sample size (10 repos) and that the corpus is small pure-Python
  repos chosen to fit a T4's ~5,300-token ceiling. The claim is scoped to that.
- Grounding filters **unverifiable** findings, not **wrong** ones. A model can
  quote a real line and still misjudge it. Say this before someone asks.